# qlora_check — consolidated results for the QLoRA arm
Reads the frozen splits and `scores/` only; generates nothing. The main score-derived arm-E result tables and diagnostics are reproduced here from the frozen score files; no generation or judging is rerun.

Rank 8 at step 2000, the GPT-4.1 val argmax. The comparison that carries the arm is against LoRA at the same rank in bf16: same data, same sampler, same adapter shape, so the delta is a selected-arm comparison at matched rank; it is not an isolated measure of the NF4 base, since the two arms sit at different checkpoints. Section 9 separates that from inference-side quantisation alone, which was measured without any adapter.

## 1. Load and integrity

In [1]:
from google.colab import drive
import os, json, math, re, itertools
from collections import defaultdict
import numpy as np
from scipy import stats
drive.mount("/content/drive")

DRIVE  = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES = f"{DRIVE}/scores"

def load_by_uid(path):
    d = {}
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: continue
        d[r["uid"]] = r
    return d

ov  = lambda r: math.sqrt(max(r["PF"], 0) * max(r["SC"], 0))
MET = [("PF", lambda r: r["PF"]), ("SC", lambda r: r["SC"]), ("Overall", ov)]

def guessed(path):
    n = 0
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if str(r.get("PF_reason", "")).startswith("guess_if_cannot_parse") \
           or str(r.get("SC_reason", "")).startswith("guess_if_cannot_parse"): n += 1
    return n

RUN, STEP = "qlora_r8", 2000
VSTEPS = (500, 1000, 1500, 2000)

BASE = load_by_uid(f"{SCORES}/test__baseline.jsonl")
Q    = load_by_uid(f"{SCORES}/test__{RUN}_step{STEP}.jsonl")
L8   = load_by_uid(f"{SCORES}/test__lora_r8_step1000.jsonl")   # same rank, bf16
L4   = load_by_uid(f"{SCORES}/test__lora_r4_step1000.jsonl")
L8_4 = load_by_uid(f"{SCORES}/test__lora_r8.jsonl")            # bf16 at step 4000

VBASE = load_by_uid(f"{SCORES}/val__baseline_val.jsonl")
VAL   = {s: load_by_uid(f"{SCORES}/val__{RUN}_step{s}.jsonl") for s in VSTEPS}
VL8   = {s: load_by_uid(f"{SCORES}/val__lora_r8_step{s}.jsonl") for s in (1000, 2000, 3000, 4000)}

# quantisation without any adapter, 24-row stratified sanity set
Q309 = load_by_uid(f"{SCORES}/test__qlora_nf4_q309_n24.jsonl")
Q302 = load_by_uid(f"{SCORES}/test__qlora_nf4_q302_n24.jsonl")

META = {r["output_image"] + "|" + "|".join(sorted(r["input_images"])): r
        for r in [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]}

I  = sorted(set(BASE) & set(Q) & set(L8) & set(L4) & set(L8_4))
IV = sorted(set(VBASE) & set.intersection(*[set(d) for d in VAL.values()]))
IQ = sorted(set(Q309) & set(Q302) & set(BASE))

files = ["test__baseline", f"test__{RUN}_step{STEP}", "test__lora_r8_step1000", "test__lora_r4_step1000",
         "test__lora_r8", "test__qlora_nf4_q309_n24", "test__qlora_nf4_q302_n24", "val__baseline_val"] \
        + [f"val__{RUN}_step{s}" for s in VSTEPS]
print(f"test rows paired across baseline, QLoRA and the bf16 adapters : {len(I)}")
print(f"val rows paired across all QLoRA checkpoints                  : {len(IV)}")
print(f"quantisation-probe rows paired                                : {len(IQ)}")
print(f"guessed judge scores across every file above                  : {sum(guessed(f'{SCORES}/{f}.jsonl') for f in files)}")

Mounted at /content/drive
test rows paired across baseline, QLoRA and the bf16 adapters : 488
val rows paired across all QLoRA checkpoints                  : 98
quantisation-probe rows paired                                : 24
guessed judge scores across every file above                  : 0


## 2. Headline — test set, 488 rows

In [2]:
def paired(A, B, idx, f):
    d = np.array([f(A[k]) - f(B[k]) for k in idx], float)
    n = len(d); se = d.std(ddof=1) / math.sqrt(n)
    p = stats.ttest_rel([f(A[k]) for k in idx], [f(B[k]) for k in idx])[1]
    tc = stats.t.ppf(0.95, n - 1)
    return dict(n=n, mean=d.mean(), sd=d.std(ddof=1), se=se, p=p,
                lo=d.mean() - tc * se, hi=d.mean() + tc * se, d=d)

def show(lab, A, B, idx, mets=MET):
    for name, f in mets:
        s = paired(A, B, idx, f)
        print(f"  {lab:32s} {name:8s} n={s['n']:4d}  delta {s['mean']:+7.3f}  sigma_d {s['sd']:5.3f}  "
              f"SE {s['se']:.3f}  p {s['p']:.3g}  90%CI [{s['lo']:+.3f}, {s['hi']:+.3f}]")

BOUND = 0.15
def tost_clustered(A, B, idx, bound=BOUND):
    """Target-clustered TOST. Robustness companion to tost(); iid remains pre-specified."""
    d = np.array([ov(A[k]) - ov(B[k]) for k in idx]); m = d.mean(); n = len(d)
    g = defaultdict(list)
    for k, x in zip(idx, d): g[k.split("|")[0]].append(x)
    G = len(g)
    se = math.sqrt(sum((sum(v) - len(v) * m) ** 2 for v in g.values()) / n**2 * (G / (G - 1)))
    tc = stats.t.ppf(0.95, G - 1)
    p_lo = 1 - stats.t.cdf((m + bound) / se, G - 1)
    p_hi = stats.t.cdf((m - bound) / se, G - 1)
    return dict(mean=m, se=se, lo=m - tc * se, hi=m + tc * se, G=G), p_lo, p_hi, (p_lo < 0.05 and p_hi < 0.05)

def tost(A, B, idx, bound=BOUND):
    s = paired(A, B, idx, ov); n = s["n"]
    p_lo = 1 - stats.t.cdf((s["mean"] + bound) / s["se"], n - 1)
    p_hi = stats.t.cdf((s["mean"] - bound) / s["se"], n - 1)
    return s, p_lo, p_hi, (p_lo < 0.05 and p_hi < 0.05)

def delta_icc(A, B, idx):
    g = defaultdict(list)
    for k in idx: g[k.split("|")[0]].append(ov(A[k]) - ov(B[k]))
    allv = np.concatenate([np.array(v) for v in g.values()]); kbar = np.mean([len(v) for v in g.values()])
    grand = allv.mean()
    msb = sum(len(v) * (np.mean(v) - grand)**2 for v in g.values()) / (len(g) - 1)
    msw = sum(((np.array(v) - np.mean(v))**2).sum() for v in g.values()) / (len(allv) - len(g))
    return (msb - msw) / (msb + (kbar - 1) * msw)

def cluster_se(A, B, idx):
    d = np.array([ov(A[k]) - ov(B[k]) for k in idx]); m = d.mean(); n = len(d)
    g = defaultdict(list)
    for k, x in zip(idx, d): g[k.split("|")[0]].append(x)
    G = len(g)
    num = sum((np.sum(np.array(v) - m))**2 for v in g.values())
    return m, d.std(ddof=1) / math.sqrt(n), math.sqrt(num / n**2 * (G / (G - 1))), G

PERSON = {"man","woman","men","women","person","people","boy","girl","boys","girls","guy","lady","businessman",
          "businesswoman","musician","musicians","child","children","kid","kids","baby","model","player","athlete",
          "worker","couple","gentleman","barber","tourist","tourists","doctor","doctors","assistant","assistants"}
_PREP = (r"\b(in|with|on|at|by|of|from|and|to|dressed|wearing|holding|using|reviewing|sitting|standing|"
         r"typing|sketching|working|reading|writing|playing|looking|smiling|drinking|reaching|leaning|posing|walking|browsing)\b")
def person_obj(instr):
    kinds = set()
    for p in re.split(r"(?<=\.)\s+", instr):
        if "<img>" not in p: continue
        m = re.search(r"(?:the\s+)?(.+?)\s+(?:is|are)\s+", p.lower())
        head = re.split(_PREP, m.group(1) if m else p.lower())[0].strip().split()
        kinds.add("person" if (head and head[-1] in PERSON) else "object")
    return "person" if kinds == {"person"} else ("object" if kinds == {"object"} else "mixed")

def strata(A, B, key, order, idx, PROV):
    print(f"  {key:12s} {'n':>4s} {'dPF':>8s} {'dSC':>8s} {'dOverall':>9s} {'p':>8s}")
    for lev in order:
        u = [k for k in idx if (PROV[k] if key == "provenance" else B[k][key]) == lev]
        if not u: continue
        dP = np.mean([A[k]["PF"] - B[k]["PF"] for k in u])
        dS = np.mean([A[k]["SC"] - B[k]["SC"] for k in u])
        d  = np.array([ov(A[k]) - ov(B[k]) for k in u])
        p  = stats.ttest_rel([ov(A[k]) for k in u], [ov(B[k]) for k in u])[1]
        print(f"  {str(lev):12s} {len(u):>4d} {dP:>+8.3f} {dS:>+8.3f} {d.mean():>+9.3f} {p:>8.3f}")

def agg(d, idx): return tuple(np.mean([f(d[k]) for k in idx]) for _, f in MET)
print(f"{'run':32s} {'PF':>7s} {'SC':>7s} {'Overall':>7s}")
for lab, d in [("prompting baseline", BASE), (f"QLoRA r8 @{STEP} (NF4)", Q),
               ("LoRA r4 @1000 (bf16)", L4), ("LoRA r8 @1000 (bf16)", L8),
               ("LoRA r8 @4000 (bf16)", L8_4)]:
    print(f"{lab:32s} " + " ".join(f"{x:7.3f}" for x in agg(d, I)))

run                                   PF      SC Overall
prompting baseline                 8.004   8.088   7.963
QLoRA r8 @2000 (NF4)               7.990   7.988   7.883
LoRA r4 @1000 (bf16)               8.027   7.855   7.848
LoRA r8 @1000 (bf16)               8.064   7.846   7.843
LoRA r8 @4000 (bf16)               8.055   7.432   7.595


## 3. Paired contrasts
The bf16 adapter at the same rank is the reference this arm is read against, each at its own GPT-selected checkpoint.

In [3]:
print("against the prompting baseline:")
show(f"QLoRA r8 @{STEP} - baseline", Q, BASE, I)
show("LoRA r8 @1000 - baseline",     L8, BASE, I)
show("LoRA r4 @1000 - baseline",     L4, BASE, I)
print("\nselected QLoRA r8 vs selected bf16 LoRA r8, matched rank:")
show("QLoRA - LoRA r8 @1000 (bf16)", Q, L8, I)
show("QLoRA - LoRA r4 @1000",        Q, L4, I)
show("QLoRA - LoRA r8 @4000",        Q, L8_4, I)

against the prompting baseline:
  QLoRA r8 @2000 - baseline        PF       n= 488  delta  -0.014  sigma_d 0.992  SE 0.045  p 0.749  90%CI [-0.088, +0.060]
  QLoRA r8 @2000 - baseline        SC       n= 488  delta  -0.100  sigma_d 1.689  SE 0.076  p 0.19  90%CI [-0.226, +0.026]
  QLoRA r8 @2000 - baseline        Overall  n= 488  delta  -0.080  sigma_d 1.376  SE 0.062  p 0.2  90%CI [-0.183, +0.023]
  LoRA r8 @1000 - baseline         PF       n= 488  delta  +0.059  sigma_d 1.004  SE 0.045  p 0.192  90%CI [-0.016, +0.134]
  LoRA r8 @1000 - baseline         SC       n= 488  delta  -0.242  sigma_d 1.524  SE 0.069  p 0.0005  90%CI [-0.356, -0.128]
  LoRA r8 @1000 - baseline         Overall  n= 488  delta  -0.120  sigma_d 1.191  SE 0.054  p 0.0269  90%CI [-0.209, -0.031]
  LoRA r4 @1000 - baseline         PF       n= 488  delta  +0.023  sigma_d 0.985  SE 0.045  p 0.614  90%CI [-0.051, +0.096]
  LoRA r4 @1000 - baseline         SC       n= 488  delta  -0.234  sigma_d 1.494  SE 0.068  p 0.00060

## 4. TOST against the pre-registered ±0.15 bound

In [4]:
for lab, A, B in [(f"QLoRA vs baseline", Q, BASE), ("QLoRA vs LoRA r8", Q, L8),
                  ("QLoRA vs LoRA r4", Q, L4), ("LoRA r8 vs baseline", L8, BASE)]:
    s, p_lo, p_hi, eq = tost(A, B, I)
    print(f"  {lab:22s} delta {s['mean']:+.3f}  90%CI [{s['lo']:+.3f}, {s['hi']:+.3f}]  "
                    f"p(>-{BOUND})={p_lo:.3f}  p(<+{BOUND})={p_hi:.2g}  -> {'equivalent' if eq else 'not equivalent'}")
    _c, _clo, _chi, _ceq = tost_clustered(A, B, I)
    print(f"  {'':22s} clustered 90%CI [{_c['lo']:+.3f}, {_c['hi']:+.3f}]  "
          f"TOST p {max(_clo,_chi):.4f}  -> {'equivalent' if _ceq else 'not equivalent'} ({_c['G']} targets)")

  QLoRA vs baseline      delta -0.080  90%CI [-0.183, +0.023]  p(>-0.15)=0.130  p(<+0.15)=0.00012  -> not equivalent
                         clustered 90%CI [-0.184, +0.024]  TOST p 0.1330  -> not equivalent (100 targets)
  QLoRA vs LoRA r8       delta +0.040  90%CI [-0.061, +0.141]  p(>-0.15)=0.001  p(<+0.15)=0.037  -> equivalent
                         clustered 90%CI [-0.056, +0.135]  TOST p 0.0290  -> equivalent (100 targets)
  QLoRA vs LoRA r4       delta +0.035  90%CI [-0.049, +0.119]  p(>-0.15)=0.000  p(<+0.15)=0.012  -> equivalent
                         clustered 90%CI [-0.039, +0.109]  TOST p 0.0056  -> equivalent (100 targets)
  LoRA r8 vs baseline    delta -0.120  90%CI [-0.209, -0.031]  p(>-0.15)=0.287  p(<+0.15)=4e-07  -> not equivalent
                         clustered 90%CI [-0.214, -0.025]  TOST p 0.2977  -> not equivalent (100 targets)


## 5. Cluster-robust standard errors

In [5]:
print(f"  {'contrast':24s} {'ICC(delta)':>10s} {'SE iid':>8s} {'SE clust':>9s} {'t iid':>7s} {'t clust':>8s} {'p clust':>8s}")
for lab, A, B in [("QLoRA - baseline", Q, BASE), ("QLoRA - LoRA r8", Q, L8), ("LoRA r8 - baseline", L8, BASE)]:
    icc = delta_icc(A, B, I); m, se_i, se_c, G = cluster_se(A, B, I)
    p = 2 * (1 - stats.norm.cdf(abs(m / se_c)))
    print(f"  {lab:24s} {icc:>10.3f} {se_i:>8.4f} {se_c:>9.4f} {m/se_i:>7.2f} {m/se_c:>8.2f} {p:>8.3f}")

  contrast                 ICC(delta)   SE iid  SE clust   t iid  t clust  p clust
  QLoRA - baseline              0.033   0.0623    0.0627   -1.28    -1.27    0.202
  QLoRA - LoRA r8               0.044   0.0615    0.0575    0.65     0.69    0.488
  LoRA r8 - baseline            0.024   0.0539    0.0568   -2.22    -2.11    0.035


## 6. Stratification

In [6]:
PROV = {k: person_obj(META[k]["instruction"]) for k in I}
print("QLoRA vs baseline:")
strata(Q, BASE, "ref_kind",   ["no_up","mixed","all_up"], I, PROV)
strata(Q, BASE, "n_refs",     [1,2,3],                    I, PROV)
strata(Q, BASE, "provenance", ["person","object","mixed"],I, PROV)
print("\nLoRA r8 @1000 vs baseline, for comparison:")
strata(L8, BASE, "ref_kind",   ["no_up","mixed","all_up"], I, PROV)

QLoRA vs baseline:
  ref_kind        n      dPF      dSC  dOverall        p
  no_up         157   +0.064   -0.121    -0.033    0.530
  mixed         133   -0.143   +0.038    -0.052    0.508
  all_up        198   +0.010   -0.177    -0.136    0.327
  n_refs          n      dPF      dSC  dOverall        p
  1             243   +0.000   -0.181    -0.132    0.217
  2             194   -0.010   -0.010    -0.013    0.848
  3              51   -0.098   -0.059    -0.085    0.627
  provenance      n      dPF      dSC  dOverall        p
  person        130   +0.054   -0.131    -0.043    0.464
  object        224   +0.013   -0.165    -0.123    0.318
  mixed         134   -0.127   +0.037    -0.043    0.574

LoRA r8 @1000 vs baseline, for comparison:
  ref_kind        n      dPF      dSC  dOverall        p
  no_up         157   +0.178   -0.115    +0.046    0.441
  mixed         133   +0.023   +0.053    +0.043    0.510
  all_up        198   -0.010   -0.540    -0.360    0.002


## 7. Validation curves and the two stopping signals
`warmup_t` is 500, so step 500 is the warmup-boundary checkpoint and has fewer optimisation steps than the later ones. Here the raw argmax and the post-warmup argmax coincide, both selecting step 2000.

In [7]:
VLOSS = {500:0.35534, 1000:0.35482, 1500:0.35471, 2000:0.35466}  # lora/experiments/qlora_r8/val_loss.jsonl
b = np.mean([ov(VBASE[k]) for k in IV])
print(f"val baseline Overall {b:.3f}")
print(f"  {'step':>5s} {'val loss':>9s} {'GPT Overall':>12s} {'GPT PF':>7s} {'GPT SC':>7s} {'vs base':>9s} {'p':>8s}")
for s in VSTEPS:
    D = VAL[s]; d = np.array([ov(D[k]) - ov(VBASE[k]) for k in IV])
    print(f"  {s:>5d} {VLOSS[s]:>9.5f} {np.mean([ov(D[k]) for k in IV]):>12.3f} "
          f"{np.mean([D[k]['PF'] for k in IV]):>7.3f} {np.mean([D[k]['SC'] for k in IV]):>7.3f} "
          f"{d.mean():>+9.3f} {stats.ttest_1samp(d, 0)[1]:>8.3f}")
g = {s: np.mean([ov(VAL[s][k]) for k in IV]) for s in VSTEPS}
post = [s for s in VSTEPS if s > 500]
print(f"  GPT argmax (all) {max(g, key=g.get)} | GPT argmax (post-warmup) {max(post, key=lambda s: g[s])} | "
      f"val-loss argmin {min(VLOSS, key=VLOSS.get)}")
print(f"  Spearman(GPT Overall, -val loss) = {stats.spearmanr([g[s] for s in VSTEPS], [-VLOSS[s] for s in VSTEPS]).statistic:+.2f}")
print("\n  pairwise between checkpoints, all six unordered pairs (is the curve rising or flat?):")
_pp = []
for a, bb in itertools.combinations(VSTEPS, 2):
    show(f"@{bb} - @{a}", VAL[bb], VAL[a], IV, mets=[("Overall", ov)])
    _pp.append(paired(VAL[bb], VAL[a], IV, ov)["p"])
print(f"\n  smallest raw p {min(_pp):.4f}; Bonferroni over the six exploratory pairs "
      f"{min(1, min(_pp)*len(_pp)):.3f}")
print("  the only raw-significant contrast crosses the warmup boundary at 500 (warmup_t = 500),")
print("  which has fewer optimisation steps than the later checkpoints. No post-warmup pairwise")
print("  difference was detected; equivalence within +/-0.15 was not established for any pair.")

val baseline Overall 7.739
   step  val loss  GPT Overall  GPT PF  GPT SC   vs base        p
    500   0.35534        7.582   7.684   7.704    -0.157    0.252
   1000   0.35482        7.747   7.837   7.786    +0.008    0.947
   1500   0.35471        7.634   7.827   7.633    -0.105    0.472
   2000   0.35466        7.784   7.888   7.816    +0.045    0.751
  GPT argmax (all) 2000 | GPT argmax (post-warmup) 2000 | val-loss argmin 2000
  Spearman(GPT Overall, -val loss) = +0.80

  pairwise between checkpoints, all six unordered pairs (is the curve rising or flat?):
  @1000 - @500                     Overall  n=  98  delta  +0.165  sigma_d 0.879  SE 0.089  p 0.0655  90%CI [+0.018, +0.313]
  @1500 - @500                     Overall  n=  98  delta  +0.052  sigma_d 0.828  SE 0.084  p 0.535  90%CI [-0.087, +0.191]
  @2000 - @500                     Overall  n=  98  delta  +0.202  sigma_d 0.903  SE 0.091  p 0.0288  90%CI [+0.051, +0.354]
  @1500 - @1000                    Overall  n=  98  delta 

## 8. Val-to-test transfer

In [8]:
dv = np.mean([ov(VAL[STEP][k]) - ov(VBASE[k]) for k in IV])
dt = np.mean([ov(Q[k]) - ov(BASE[k]) for k in I])
print(f"  QLoRA @{STEP} - baseline : val {dv:+.3f} (n={len(IV)})   test {dt:+.3f} (n={len(I)})   "
      f"same sign {np.sign(dv)==np.sign(dt)}")
kv = sorted(set(VAL[1000]) & set(VL8[1000]))
dv8 = np.mean([ov(VAL[1000][k]) - ov(VL8[1000][k]) for k in kv])
dt8 = np.mean([ov(Q[k]) - ov(L8[k]) for k in I])
print(f"  QLoRA - LoRA r8         : val {dv8:+.3f} (matched step 1000)   test {dt8:+.3f} (best vs best)")
print("  the val slice ranks checkpoints; it does not size effects")

  QLoRA @2000 - baseline : val +0.045 (n=98)   test -0.080 (n=488)   same sign False
  QLoRA - LoRA r8         : val -0.196 (matched step 1000)   test +0.040 (best vs best)
  the val slice ranks checkpoints; it does not size effects


## 9. Quantisation without an adapter
Isolates the NF4 base from anything the adapter learned. `q309` quantises all 309 linear modules; `q302` leaves seven bottleneck modules in bf16. 24-row stratified sanity set, 8 cells x 3 rows.

In [9]:
print(f"{'run':34s} {'PF':>7s} {'SC':>7s} {'Overall':>7s}  (n={len(IQ)})")
for lab, d in [("bf16 base (no adapter)", BASE), ("NF4 all 309 modules", Q309), ("NF4 minus 7 bottlenecks", Q302)]:
    print(f"{lab:34s} " + " ".join(f"{np.mean([f(d[k]) for k in IQ]):7.3f}" for _, f in MET))
print()
show("q309 - bf16 base", Q309, BASE, IQ)
show("q302 - q309",      Q302, Q309, IQ)

run                                     PF      SC Overall  (n=24)
bf16 base (no adapter)               7.875   8.333   8.078
NF4 all 309 modules                  7.792   8.375   8.003
NF4 minus 7 bottlenecks              7.708   7.875   7.723

  q309 - bf16 base                 PF       n=  24  delta  -0.083  sigma_d 1.248  SE 0.255  p 0.747  90%CI [-0.520, +0.353]
  q309 - bf16 base                 SC       n=  24  delta  +0.042  sigma_d 0.955  SE 0.195  p 0.833  90%CI [-0.292, +0.376]
  q309 - bf16 base                 Overall  n=  24  delta  -0.074  sigma_d 0.896  SE 0.183  p 0.688  90%CI [-0.388, +0.239]
  q302 - q309                      PF       n=  24  delta  -0.083  sigma_d 1.060  SE 0.216  p 0.704  90%CI [-0.454, +0.287]
  q302 - q309                      SC       n=  24  delta  -0.500  sigma_d 1.103  SE 0.225  p 0.0366  90%CI [-0.886, -0.114]
  q302 - q309                      Overall  n=  24  delta  -0.281  sigma_d 0.965  SE 0.197  p 0.168  90%CI [-0.618, +0.057]


## 10. Rank coverage
The spec lists QLoRA at ranks 4/8/16. Only rank 8 was run. The bound below is what the LoRA arm already establishes about rank on this pipeline, and it is the quantity the missing cells would have to overturn.

In [10]:
K1 = {r: load_by_uid(f"{SCORES}/test__lora_r{r}_step1000.jsonl") for r in (4, 8, 16)}
K4 = {r: load_by_uid(f"{SCORES}/test__lora_r{r}.jsonl")            for r in (4, 8, 16)}
J  = sorted(set(BASE) & set.intersection(*[set(d) for d in list(K1.values()) + list(K4.values())]))
# see lora_check: the contrasts within a checkpoint are algebraically dependent and the
# rank relationship is not monotone, so each pairwise contrast is reported on its own
for D, lab in ((K1, "@1000 (selected)"), (K4, "@4000")):
    widest = 0.0
    for a, b in itertools.combinations((4, 8, 16), 2):
        s = paired(D[b], D[a], J, ov)
        widest = max(widest, abs(s["mean"]) + 1.96 * s["se"])   # 95% bound; TOST decides on the 90% CI, 0.093 at step 1000
        print(f"  r{a}->r{b:<3} {lab:<18} delta {s['mean']:+.4f}  SE {s['se']:.4f}")
    print(f"  {lab}: no rank contrast significant; widest 95% pairwise bound {widest:.3f}")
print("  the bf16 LoRA sweep establishes rank equivalence at step 1000; QLoRA rank sensitivity is\n"
      "  unmeasured, and r4/r16 are deferred as a scope decision, not settled by the bf16 result")

  r4->r8   @1000 (selected)   delta -0.0051  SE 0.0435
  r4->r16  @1000 (selected)   delta +0.0037  SE 0.0395
  r8->r16  @1000 (selected)   delta +0.0088  SE 0.0509
  @1000 (selected): no rank contrast significant; widest 95% pairwise bound 0.109
  r4->r8   @4000              delta +0.0789  SE 0.0603
  r4->r16  @4000              delta +0.0105  SE 0.0687
  r8->r16  @4000              delta -0.0684  SE 0.0537
  @4000: no rank contrast significant; widest 95% pairwise bound 0.197
  the bf16 LoRA sweep establishes rank equivalence at step 1000; QLoRA rank sensitivity is
  unmeasured, and r4/r16 are deferred as a scope decision, not settled by the bf16 result
